# MHC-I display-window status — is the window ON in cancer cells? (RUNG-18 + 18b)

The whole **immune route** assumes a cancer cell keeps its MHC-I "display window" lit so a T-cell can read the neoantigen. This notebook MEASURES that, two ways:

1. **RUNG-18 (genetic, CPU, seconds):** across **6,319 WGS tumours** — how often the window is genetically *broken* (B2M/TAP loss = route dies) vs *dimmed* (HLA-LOH = route survives) vs intact.
2. **RUNG-18b (expression, Census, ~15–25 min):** in **real malignant cells** — how often the window is *switched off without being broken* (epigenetic/transcriptional silencing the genetics can't see). Housekeeping-gated for dropout; immune cells as a positive control.

**Ceiling:** mRNA ≠ surface protein; HLA-I is IFN-inducible (resting atlas over-states silencing). Genetic-dark = permanent floor; transcriptional-dark = the (often reversible) addition. Not a wet result.

In [ ]:
#@title Cell 1 — clone / pull repo
import os
from pathlib import Path
REPO = Path('/content/cancer-recog-apoptosis')
if REPO.exists():
    !cd {REPO} && rm -f runs/logs/*.log && git fetch origin && git reset --hard origin/main
else:
    !git clone https://github.com/AnshumanAtrey/cancer-recog-apoptosis.git {REPO}
os.chdir(REPO)
!git log --oneline -1
print('[CELL 1] ✓')

In [ ]:
#@title Cell 2 — run log
from scripts.runlog import new_log
RUNLOG = new_log('rung18_mhc_window', repo_dir='.')
print('[CELL 2] ✓ logging to', RUNLOG)

In [ ]:
#@title Cell 3 — VALIDATE ports (selftest). Stop if anything fails.
import sys
from scripts.runlog import run_logged
rcs = [run_logged([sys.executable,'-u',f'scripts/{s}','selftest'], RUNLOG) for s in
       ('43_mhc_window_status.py','44_mhc_window_expression.py')]
print('[CELL 3]', '✓ all validated' if all(r==0 for r in rcs) else f'✗ FAILED {rcs} — stop here')

In [ ]:
#@title Cell 4 — RUNG-18 (genetic): window status across 6,319 WGS tumours  [CPU, seconds]
import sys, os, json
from scripts.runlog import run_logged
run_logged([sys.executable,'-u','scripts/43_mhc_window_status.py','run'], RUNLOG)
from IPython.display import Image, display
p='runs/rung18_mhc_window/rung18_mhc_window'
if os.path.exists(p+'.png'): display(Image(p+'.png'))
if os.path.exists(p+'.json'):
    d=json.load(open(p+'.json')); print('HEADLINE:', json.dumps(d['HEADLINE'], indent=2))

In [ ]:
#@title Cell 5 — (Census) RUNG-18b (expression): is the window SWITCHED OFF in real malignant cells?  [~15–25 min]
#@markdown HLA-A/B/C + B2M transcription in malignant cells, housekeeping-gated for dropout, immune cells as control.
import os, sys, importlib.util
from scripts.runlog import run_logged
try:
    from google.colab import drive; drive.mount('/content/drive')
    cd='/content/drive/MyDrive/cancer-recon'; os.makedirs(cd, exist_ok=True)
    os.environ['RUNG18B_CACHE']=f'{cd}/rung18b_tiles'; os.makedirs(os.environ['RUNG18B_CACHE'], exist_ok=True)
    print('[CELL 5] resumable tiles ->', os.environ['RUNG18B_CACHE'])
except Exception as e:
    print('[CELL 5] no Drive (', type(e).__name__, ') — tiles local, not resumable across disconnects')
for pk,pn in [('cellxgene_census','cellxgene-census'),('scanpy','scanpy')]:
    if importlib.util.find_spec(pk) is None:
        run_logged([sys.executable,'-m','pip','install','-q',pn], RUNLOG, label=f'pip {pn}')
rc = run_logged([sys.executable,'-u','scripts/44_mhc_window_expression.py','run'], RUNLOG)
from IPython.display import Image, display
import json
p='runs/rung18b_mhc_expression/rung18b_mhc_expression'
if os.path.exists(p+'.png'): display(Image(p+'.png'))
if os.path.exists(p+'.json'):
    d=json.load(open(p+'.json'))
    print('per_route_bucket:\n', json.dumps(d.get('per_route_bucket',{}), indent=2)[:1800])
    print('\ngenetic_floor_from_rung18:\n', json.dumps(d.get('genetic_floor_from_rung18',{}), indent=2))
print('[CELL 5]', '✓ done' if rc==0 else f'(exit {rc}; re-run to resume, or skip)')

In [ ]:
#@title Cell 6 — bundle ONE zip + download
import os, glob, zipfile
from scripts.runlog import finalize
finalize(RUNLOG, download=False)
stem = os.path.basename(str(RUNLOG)).replace('rung18_mhc_window_','').replace('.log','')
b = f'/content/rung18_mhc_run_{stem}.zip'
ps = sorted(set(sum([glob.glob(f'runs/{d}/*.png')+glob.glob(f'runs/{d}/*.json')
                     for d in ('rung18_mhc_window','rung18b_mhc_expression')], [])+[str(RUNLOG)]))
with zipfile.ZipFile(b,'w',zipfile.ZIP_DEFLATED) as z:
    for x in ps:
        if os.path.exists(x): z.write(x, arcname=os.path.basename(x)); print(' bundled', x)
print('[CELL 6] ->', b)
try:
    from google.colab import files; files.download(b)
except Exception as e: print('(download skipped:', e, ')')